# Step 2 Diagnostics: RBH
How RBH's IDs link, which PICANet file is right, what the times mean, and whether the drug tables thin out after 2018. Counts only. The decisions these support are used by the manifest (step 3); nothing here changes any data.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import diagnostics as D
import manifest as M
import picanet_times as PT
import rbh_diagnostics as RD

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
reports_dir = open_picu_dir / "reports"
rbh_dir = base / "RBH"

d = RD.load(rbh_dir)       # demographics and the vitals key and time columns, read once
rbh_m, rbh_w = M.rbh_manifest(rbh_dir)

## 1. How the IDs link
A studyID is prefix : PATIENT_RK : admission time (SAS seconds). The prefix is NIC or NND; the same stay can carry either. `placeholder_rows` shows why old stays appear in every table: the extract starts from demographics, so a stay with no data is one row with an empty chartTime.

In [ ]:
D.show(RD.id_hierarchy(d))

In [ ]:
D.show(RD.study_id_structure(d))

In [ ]:
D.show(RD.empty_studies(d))

In [ ]:
RD.placeholder_rows(d)

## 2. Which PICANet file, and the NIC/NND twins
The manifest uses the new PICANet file. These compare it with the old one at stay level (prefix removed), count stays that appear under both prefixes in each file, and check how alike the two copies are in sedation.

In [ ]:
D.show(RD.picanet_versions(rbh_dir, d))

In [ ]:
RD.picanet_vs_picu_rows(d, rbh_dir)

In [ ]:
RD.prefix_twins(rbh_dir)

In [ ]:
RD.sedation_twins(rbh_dir)

In [ ]:
RD.encounter_keys(rbh_dir)

## 3. Times
Which time is the admission, and how the ward times, `intv_hr` and LOS relate. The demographics-versus-PICANet time and LOS checks are in `picanet_times.py`.

In [ ]:
D.show(RD.vitals_times(d))

In [ ]:
D.show(RD.multi_study_admissions(d))

In [ ]:
PT.check_times(rbh_dir, "RBH", demo_adm="PICUadm_dttm", los_col="LOSHours", los_unit="hours", manifest=rbh_m)

## 4. Do the drug tables thin out after 2018?
For the eligible stays, the share with at least one row in each drug table, by admission year. A fall in the share is missing data; a flat share with fewer rows is a change in charting.

In [ ]:
RD.drug_coverage_by_year(rbh_dir, rbh_m)

In [ ]:
RD.drug_by_year(rbh_dir, rbh_m)

In [ ]:
RD.rows_by_month(rbh_dir, "ai_picu_infusion.csv", start="2018-06", end="2019-12")

## 5. Save
Writes every table above to one dated report.

In [ ]:
D.write_report({
    "IDs": RD.id_hierarchy(d), "ID structure": RD.study_id_structure(d),
    "Empty studies": RD.empty_studies(d), "Placeholder rows": RD.placeholder_rows(d),
    "PICANet versions": RD.picanet_versions(rbh_dir, d), "Twins": RD.prefix_twins(rbh_dir),
    "Sedation twins": RD.sedation_twins(rbh_dir), "Times": RD.vitals_times(d),
    "Drug coverage by year": RD.drug_coverage_by_year(rbh_dir, rbh_m),
    "Drug by year (infusion)": RD.drug_by_year(rbh_dir, rbh_m),
    "Infusion rows by month": RD.rows_by_month(rbh_dir, "ai_picu_infusion.csv", start="2018-06", end="2019-12"),
    "PICANet old vs new": RD.picanet_old_vs_new(rbh_dir, d),
}, reports_dir / f"RBH_diagnostics_{pd.Timestamp.today().date()}.md")

In [ ]:
D.show(RD.picanet_old_vs_new(rbh_dir, d))